#  Analyse Exploratoire des Accidents de la Route aux États-Unis (EDA)

## Objectifs
Cette phase vise à :
- Vérifier la qualité globale des données brutes

- Traiter les valeurs manquantes (imputation, suppression ou réécriture)

- Nettoyer les incohérences temporelles, géographiques et météorologiques

- Harmoniser les types de variables (dates, catégories, booléens, flottants…)

Les données ont été préalablement nettoyées et imputées afin de garantir la fiabilité des analyses.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# ======================
# Dtypes optimisés
# ======================
dtype_map = {
    "ID": "string",
    "Severity": "int8",
    "Start_Lat": "float32",
    "Start_Lng": "float32",
    "End_Lat": "float32",
    "End_Lng": "float32",
    "Distance(mi)": "float32",
    "City": "category",
    "County": "category",
    "State": "category",
    "Wind_Direction": "category",
    "Weather_Condition": "category",
    "Sunrise_Sunset": "category",
    "Civil_Twilight": "category",
}

bool_cols = [
    "Amenity","Bump","Crossing","Give_Way","Junction","No_Exit",
    "Railway","Roundabout","Station","Stop","Traffic_Calming","Traffic_Signal"
]

drop_cols = [
    "Country", "Source", "Turning_Loop", "Description", "Street",
    "Zipcode", "Timezone", "Nautical_Twilight", "Astronomical_Twilight"
]

In [ ]:
df = pd.read_csv(
    "us.csv",
    dtype=dtype_map,
    parse_dates=["Start_Time", "End_Time", "Weather_Timestamp"]
)

In [3]:
for col in bool_cols:
    df[col] = df[col].astype("boolean")

In [4]:
df["Temperature(C)"] = (df["Temperature(F)"] - 32) * 5/9
df["Wind_Chill(C)"] = (df["Wind_Chill(F)"] - 32) * 5/9
df.drop(columns=["Temperature(F)", "Wind_Chill(F)"], errors="ignore", inplace=True)

In [5]:
df = df.sort_values(["Airport_Code", "Start_Time"])
df["Weather_Timestamp"] = df.groupby("Airport_Code")["Weather_Timestamp"].ffill()

In [6]:
cols_to_impute = ["Temperature(C)", "Humidity(%)", "Pressure(in)",
                  "Visibility(mi)", "Wind_Speed(mph)"]

df["Wind_Chill(C)"] = df["Wind_Chill(C)"].fillna(0)
df["Precipitation(in)"] = df["Precipitation(in)"].fillna(0)

imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=30, max_depth=6, n_jobs=-1),
    max_iter=5,
    random_state=42
)

df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])


c:\Users\HAJAR\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [7]:
for c in cols_to_impute + ["Wind_Chill(C)", "Precipitation(in)"]:
    df[c] = df[c].astype("float32")

print("✅ Nettoyage + Imputation COMPLET effectué.")

✅ Nettoyage + Imputation COMPLET effectué.


In [8]:
# Résumé des colonnes
pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.values,
    "Missing Values": df.isna().sum().values
})

,Column,Dtype,Missing Values
0,ID,string[python],0
1,Source,object,0
2,Severity,int8,0
3,Start_Time,datetime64[ns],0
4,End_Time,datetime64[ns],0
5,Start_Lat,float32,0
6,Start_Lng,float32,0
7,End_Lat,float32,500000
8,End_Lng,float32,500000
9,Distance(mi),float32,0


In [9]:
# Colonnes catégorielles
cat_cols = ["City", "Airport_Code", "Wind_Direction", "Weather_Condition",
            "Sunrise_Sunset", "Civil_Twilight"]

for col in cat_cols:
    df[col] = df[col].astype("category")
    if "Unknown" not in df[col].cat.categories:
        df[col] = df[col].cat.add_categories("Unknown")
    df[col] = df[col].fillna("Unknown")

print("✅ Colonnes catégorielles nettoyées et valeurs manquantes remplacées par 'Unknown'.")

# -------------------------------
# 2️⃣ Forward fill pour Weather_Timestamp
# -------------------------------
# On trie par Airport_Code puis Start_Time pour que le forward fill ait du sens
df = df.sort_values(["Airport_Code", "Start_Time"])

# Remplir les timestamps manquants avec la dernière valeur connue de la même station
df["Weather_Timestamp"] = df.groupby("Airport_Code")["Weather_Timestamp"].ffill()

# Vérification rapide
print("✅ Forward fill effectué pour Weather_Timestamp.")
print(df[["Airport_Code", "Start_Time", "Weather_Timestamp"]].head(10))


✅ Colonnes catégorielles nettoyées et valeurs manquantes remplacées par 'Unknown'.
✅ Forward fill effectué pour Weather_Timestamp.
       Airport_Code          Start_Time   Weather_Timestamp
295462         K11R 2016-08-18 08:20:31 2016-08-18 08:15:00
291275         K11R 2016-10-16 11:29:41 2016-10-16 11:35:00
263417         K11R 2016-12-10 07:42:13 2016-12-10 07:35:00
263418         K11R 2016-12-10 07:42:51 2016-12-10 07:35:00
263439         K11R 2016-12-10 10:43:07 2016-12-10 10:35:00
272362         K11R 2017-01-22 07:19:54 2017-01-22 07:15:00
332050         K11R 2017-01-28 12:49:41 2017-01-28 12:55:00
332066         K11R 2017-01-28 16:02:04 2017-01-28 15:55:00
216887         K12N 2016-06-21 23:43:57 2016-06-21 23:54:00
217094         K12N 2016-06-23 10:04:15 2016-06-23 09:54:00


C:\Users\HAJAR\AppData\Local\Temp\ipykernel_5344\2430968596.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["Weather_Timestamp"] = df.groupby("Airport_Code")["Weather_Timestamp"].ffill()


In [10]:
df.to_csv("us_cleaned.csv", index=False)